# Ã‰tape 2 â€” GraphSAGE vs XGBoost sur YelpChi (Colab)
RÃ©pond Ã  la question de recherche : les relations (GraphSAGE) battent-elles l'approche isolÃ©e (XGBoost) ? DGL requis (Colab). Remplacez `<your-username>`.

In [ ]:
!pip -q install torch_geometric dgl xgboost scikit-learn
!git clone https://github.com/<your-username>/pfe-fraude-graph-ml.git || true
%cd pfe-fraude-graph-ml

In [ ]:
from src.config import TrainConfig, set_seed
from src.data.yelpchi import load_yelpchi
from src.models.graphsage import GraphSAGE
from src.train.train_gnn import train_gnn, predict_scores, class_weights_from_labels
from src.train.baseline_xgb import train_xgb, predict_xgb
from src.eval.metrics import compute_metrics, print_comparison
import numpy as np

set_seed(42)
data = load_yelpchi()
y = data.y.numpy()
train = data.train_mask.numpy()
test = data.test_mask.numpy()

# GraphSAGE (relations)
cw = class_weights_from_labels(data.y[data.train_mask])
gnn = GraphSAGE(data.x.shape[1], 64, 2, dropout=0.5)
gnn = train_gnn(gnn, data, TrainConfig(epochs=200), class_weight=cw)
gnn_scores = predict_scores(gnn, data)

# XGBoost (mÃªmes features, SANS graphe)
X = data.x.numpy()
n_neg = (y[train] == 0).sum()
n_pos = (y[train] == 1).sum()
xgb = train_xgb(X[train], y[train], scale_pos_weight=n_neg / n_pos)
xgb_scores = predict_xgb(xgb, X)

# Comparaison (gabarit PC-GNN)
results = {
    "GraphSAGE": compute_metrics(y[test], gnn_scores[test]),
    "XGBoost":   compute_metrics(y[test], xgb_scores[test]),
}
print_comparison(results)